# Commission regex pipeline v4 — clean-manifest driven

This notebook scans **all successfully processed Commission documents** from the four extraction pipelines:

- `com_manual`
- `com_legacy`
- `com_antitrust`
- `com_db_comp`

The clean-file manifests are the source of truth. **`com_final_commission.xlsx` is not used.**

## Pipeline

```text
4 clean-file manifests
        ↓
standardise one row per processed document
        ↓
coverage / file-existence audit
        ↓
select canonical regex input (.txt preferred, .md fallback)
        ↓
regex scan
        ↓
1. Court cases
2. Commission references
3. Commission cases
```

### The three regex families

1. **Court cases** — Court case numbers such as `6/64`, `85/76`, `C-413/14`, `T-342/99`.
2. **Commission references** — published Commission decision references such as `88/22/EEC`, `2003/706/EC`.
3. **Commission cases** — Commission competition case identifiers such as `IV/33.322`, `COMP/37.792`, `AT.39740`.

Each main output is one row per **source Commission document × normalized matched identifier**, with occurrence counts, context, paragraph and extraction provenance. Raw occurrence-level hits are also retained.


In [1]:
from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# Excel-safe export helpers
# ---------------------------------------------------------------------
_EXCEL_ILLEGAL_CHARS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")


def excel_safe_value(value: Any) -> Any:
    if not isinstance(value, str):
        return value
    value = value.replace("\x0c", "\n")
    return _EXCEL_ILLEGAL_CHARS_RE.sub("", value)


def excel_safe_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].map(excel_safe_value)
    return out


def to_excel_safe(df: pd.DataFrame, writer, *, sheet_name: str, index: bool = False) -> None:
    excel_safe_df(df).to_excel(writer, sheet_name=sheet_name, index=index)


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
try:
    NOTEBOOK_DIR = Path.cwd().resolve()
except Exception:
    NOTEBOOK_DIR = Path('.').resolve()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if candidate.name == 'notebooks' and candidate.parent.name == 'code':
            return candidate.parent.parent
        if candidate.name == 'eccjeu':
            return candidate
        if (candidate / 'data').exists() and (candidate / 'output').exists():
            return candidate
        if (candidate / 'input').exists() and (candidate / 'output').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'commission_regex'
EXCEL_DIR = OUTPUT_DIR / 'excel'
CSV_DIR = OUTPUT_DIR / 'csv'
EXCEL_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NAME = 'run_commission_regex_v4.ipynb'

CLEAN_MANIFEST_SPECS = {
    'com_manual': {
        'csv': PROJECT_ROOT / 'output' / 'com_manual' / 'com_manual_clean_file_manifest.csv',
        'xlsx': PROJECT_ROOT / 'output' / 'com_manual' / 'com_manual_clean_file_manifest.xlsx',
    },
    'com_legacy': {
        'csv': PROJECT_ROOT / 'output' / 'com_legacy' / 'com_legacy_clean_file_manifest.csv',
        'xlsx': PROJECT_ROOT / 'output' / 'com_legacy' / 'com_legacy_clean_file_manifest.xlsx',
    },
    'com_antitrust': {
        'csv': PROJECT_ROOT / 'output' / 'com_antitrust' / 'com_antitrust_clean_file_manifest.csv',
        'xlsx': PROJECT_ROOT / 'output' / 'com_antitrust' / 'com_antitrust_clean_file_manifest.xlsx',
    },
    'com_db_comp': {
        'csv': PROJECT_ROOT / 'output' / 'com_db_comp' / 'db_comp_clean_file_manifest.csv',
        'xlsx': PROJECT_ROOT / 'output' / 'com_db_comp' / 'db_comp_clean_file_manifest.xlsx',
    },
}

INPUT_MANIFEST_XLSX = EXCEL_DIR / 'commission_regex_input_manifest.xlsx'
INPUT_MANIFEST_CSV = CSV_DIR / 'commission_regex_input_manifest.csv'
AUDIT_XLSX = EXCEL_DIR / 'commission_regex_audit.xlsx'
ALL_RAW_HITS_CSV = CSV_DIR / 'commission_regex_all_raw_hits.csv'
COURT_CASES_XLSX = EXCEL_DIR / 'commission_regex_court_cases.xlsx'
COMMISSION_REFERENCES_XLSX = EXCEL_DIR / 'commission_regex_commission_references.xlsx'
COMMISSION_CASES_XLSX = EXCEL_DIR / 'commission_regex_commission_cases.xlsx'
COURT_CASES_CSV = CSV_DIR / 'commission_regex_court_cases.csv'
COMMISSION_REFERENCES_CSV = CSV_DIR / 'commission_regex_commission_references.csv'
COMMISSION_CASES_CSV = CSV_DIR / 'commission_regex_commission_cases.csv'

# ---------------------------------------------------------------------
# Processing switches
# ---------------------------------------------------------------------
REQUIRE_ALL_MANIFESTS = True
REQUIRE_CLEAN_SUCCESS = True
INPUT_PREFERENCE = ('txt', 'md')
MAX_DOCUMENTS_PER_SOURCE = None  # e.g. 100 for a quick test; None = all
MIN_TEXT_CHARS = 20
CONTEXT_WINDOW_CHARS = 220
COMMISSION_DECISION_CONTEXT_CHARS = 50
DECISION_CONTEXT_RADIUS = 150
MAX_PARAGRAPH_CHARS = 2800
MAX_UNIQUE_VALUES = 100

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)


PROJECT_ROOT: /home/edik/projects/eccjeu
OUTPUT_DIR: /home/edik/projects/eccjeu/output/commission_regex


## 1. Load and standardise the four clean-file manifests

The extraction manifest row is the unit of input. Source-specific metadata are mapped onto common fields but the important original identifiers are retained.

The stable regex document key is:

```text
source_document_key = <commission_source> || <document_id>
```

No attempt is made here to collapse documents from different Commission sources into a single final decision record.


In [2]:
def clean_scalar(value: Any) -> str:
    if value is None:
        return ''
    try:
        if pd.isna(value):
            return ''
    except Exception:
        pass
    text = str(value).strip()
    return '' if text.lower() in {'', 'nan', 'none', 'null', '<na>'} else text


def clean_bool(value: Any, default: bool = False) -> bool:
    if isinstance(value, bool):
        return value
    text = clean_scalar(value).lower()
    if text in {'1', 'true', 'yes', 'y', 't'}:
        return True
    if text in {'0', 'false', 'no', 'n', 'f'}:
        return False
    return default


def first_nonempty(row: pd.Series, candidates: Iterable[str]) -> str:
    for col in candidates:
        if col in row.index:
            value = clean_scalar(row.get(col, ''))
            if value:
                return value
    return ''


def unique_join(values: Iterable[Any], max_values: int = MAX_UNIQUE_VALUES) -> str:
    out, seen = [], set()
    for value in values:
        value = clean_scalar(value)
        if not value or value in seen:
            continue
        seen.add(value)
        out.append(value)
        if len(out) >= max_values:
            break
    return ' | '.join(out)


def candidate_resolutions(path_value: Any) -> List[Path]:
    raw = clean_scalar(path_value)
    if not raw:
        return []
    p = Path(raw).expanduser()
    candidates = [p]
    if not p.is_absolute():
        candidates.append(PROJECT_ROOT / p)

    normalized = raw.replace('\\', '/')
    if '/eccjeu/' in normalized:
        candidates.append(PROJECT_ROOT / normalized.split('/eccjeu/', 1)[1])
    elif normalized.startswith('eccjeu/'):
        candidates.append(PROJECT_ROOT / normalized[len('eccjeu/'):])

    out, seen = [], set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve(strict=False)
        except Exception:
            pass
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            out.append(candidate)
    return out


def resolve_existing_file(path_value: Any) -> Tuple[str, bool]:
    candidates = candidate_resolutions(path_value)
    for p in candidates:
        if p.exists() and p.is_file():
            return str(p), True
    return (str(candidates[0]) if candidates else ''), False


def choose_manifest_file(spec: Dict[str, Path]) -> Optional[Path]:
    if spec['csv'].exists():
        return spec['csv']
    if spec['xlsx'].exists():
        return spec['xlsx']
    return None


def load_one_clean_manifest(source: str, spec: Dict[str, Path]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    chosen = choose_manifest_file(spec)
    status = {
        'commission_source': source,
        'manifest_found': chosen is not None,
        'manifest_path': str(chosen) if chosen else '',
        'n_rows_loaded': 0,
        'n_unique_document_ids': 0,
    }
    if chosen is None:
        return pd.DataFrame(), status

    if chosen.suffix.lower() == '.csv':
        df = pd.read_csv(chosen, low_memory=False)
    else:
        df = pd.read_excel(chosen)
    df.columns = [str(c).strip() for c in df.columns]

    if MAX_DOCUMENTS_PER_SOURCE is not None:
        df = df.head(int(MAX_DOCUMENTS_PER_SOURCE)).copy()

    if 'document_id' not in df.columns:
        raise ValueError(f'{source}: clean manifest lacks required document_id column: {chosen}')

    df['commission_source'] = source
    df['clean_manifest_path'] = str(chosen)
    df['document_id'] = df['document_id'].map(clean_scalar)
    df['source_document_key'] = source + '||' + df['document_id']

    # Common source metadata. Original source-specific identifier columns remain below as well.
    df['source_title'] = df.apply(lambda r: first_nonempty(r, [
        'decision_title', 'title_for_processing', 'title', 'item_label', 'label'
    ]), axis=1)
    df['source_date'] = df.apply(lambda r: first_nonempty(r, [
        'decision_date', 'item_date', 'document_date', 'date'
    ]), axis=1)
    df['source_case_numbers'] = df.apply(lambda r: first_nonempty(r, [
        'case_numbers_normalized', 'case_numbers_raw', 'case_numbers', 'case_number'
    ]), axis=1)
    df['source_celex'] = df.apply(lambda r: first_nonempty(r, [
        'celex', 'celex_id', 'celex_number'
    ]), axis=1)
    df['source_native_id'] = df.apply(lambda r: first_nonempty(r, [
        'dbcomp_document_id', 'source_entry_id', 'source_record_id', 'download_key',
        'identifier', 'source_native_id'
    ]), axis=1)

    # Eligibility: use the extraction notebook's own flag when present.
    def eligibility(r: pd.Series) -> bool:
        if 'cleaning_target' in r.index and clean_scalar(r.get('cleaning_target', '')):
            return clean_bool(r.get('cleaning_target'))
        if 'processing_eligible' in r.index and clean_scalar(r.get('processing_eligible', '')):
            return clean_bool(r.get('processing_eligible'))
        return True

    df['manifest_processing_eligible'] = df.apply(eligibility, axis=1)
    if 'clean_success' in df.columns:
        df['manifest_clean_success'] = df['clean_success'].map(lambda x: clean_bool(x, default=False))
    else:
        df['manifest_clean_success'] = False

    for col in ['raw_file_path', 'clean_text_path', 'readable_text_path', 'markdown_path']:
        if col not in df.columns:
            df[col] = ''

    status['n_rows_loaded'] = int(len(df))
    status['n_unique_document_ids'] = int(df['document_id'].nunique())
    return df, status


loaded = []
manifest_status_rows = []
for source, spec in CLEAN_MANIFEST_SPECS.items():
    df, status = load_one_clean_manifest(source, spec)
    manifest_status_rows.append(status)
    if not df.empty:
        loaded.append(df)

clean_manifest_status = pd.DataFrame(manifest_status_rows)
missing_sources = clean_manifest_status.loc[~clean_manifest_status['manifest_found'], 'commission_source'].tolist()

if missing_sources and REQUIRE_ALL_MANIFESTS:
    raise FileNotFoundError(
        'Missing required clean-file manifests for: ' + ', '.join(missing_sources) + '\n' +
        clean_manifest_status.to_string(index=False)
    )
if not loaded:
    raise FileNotFoundError('No clean-file manifests were found.')

# Union of columns preserves source-specific metadata without forcing one source schema onto another.
unified_manifest = pd.concat(loaded, ignore_index=True, sort=False)
unified_manifest['_loaded_row_id'] = range(1, len(unified_manifest) + 1)

# Detect duplicate source+document keys. Keep the last row for scanning, but audit duplicates explicitly.
dup_counts = unified_manifest.groupby('source_document_key', dropna=False).size().rename('manifest_duplicate_count')
unified_manifest['manifest_duplicate_count'] = unified_manifest['source_document_key'].map(dup_counts).fillna(1).astype(int)
duplicate_manifest_rows = unified_manifest[unified_manifest['manifest_duplicate_count'] > 1].copy()

if 'processed_at' in unified_manifest.columns:
    unified_manifest['_processed_sort'] = pd.to_datetime(unified_manifest['processed_at'], errors='coerce')
    unified_manifest = unified_manifest.sort_values(['source_document_key', '_processed_sort', '_loaded_row_id'])
else:
    unified_manifest = unified_manifest.sort_values(['source_document_key', '_loaded_row_id'])

regex_document_manifest = unified_manifest.drop_duplicates('source_document_key', keep='last').copy().reset_index(drop=True)
regex_document_manifest['regex_document_id'] = range(1, len(regex_document_manifest) + 1)

print('Clean-manifest status:')
display(clean_manifest_status)
print('Rows across manifests:', len(unified_manifest))
print('Unique source_document_key rows for regex:', len(regex_document_manifest))
print('Duplicate source_document_key rows found:', len(duplicate_manifest_rows))


Clean-manifest status:


,commission_source,manifest_found,manifest_path,n_rows_loaded,n_unique_document_ids
0,com_manual,True,/home/edik/projects/eccjeu/output/com_manual/c...,16,16
1,com_legacy,True,/home/edik/projects/eccjeu/output/com_legacy/c...,1505,1505
2,com_antitrust,True,/home/edik/projects/eccjeu/output/com_antitrus...,1174,1174
3,com_db_comp,True,/home/edik/projects/eccjeu/output/com_db_comp/...,833,833


Rows across manifests: 3528
Unique source_document_key rows for regex: 3528
Duplicate source_document_key rows found: 0


## 2. Coverage audit and exact regex input selection

`.txt` is preferred. `.md` is a fallback and its YAML-style extraction front matter is stripped before scanning.

A row is fed into regex only when:

- it is a processing target / eligible document;
- `clean_success=True` when `REQUIRE_CLEAN_SUCCESS=True`;
- at least one canonical `.txt` or `.md` file exists.

The audit is deliberately based on the **clean manifests themselves**. It does not rediscover raw downloads.


In [3]:
def build_input_audit(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    resolved_txt, txt_exists = [], []
    resolved_md, md_exists = [], []
    resolved_readable, readable_exists = [], []

    for _, row in out.iterrows():
        p, ok = resolve_existing_file(row.get('clean_text_path', ''))
        resolved_txt.append(p); txt_exists.append(ok)
        p, ok = resolve_existing_file(row.get('markdown_path', ''))
        resolved_md.append(p); md_exists.append(ok)
        p, ok = resolve_existing_file(row.get('readable_text_path', ''))
        resolved_readable.append(p); readable_exists.append(ok)

    out['resolved_clean_text_path'] = resolved_txt
    out['clean_text_exists'] = txt_exists
    out['resolved_markdown_path'] = resolved_md
    out['markdown_exists'] = md_exists
    out['resolved_readable_text_path'] = resolved_readable
    out['readable_text_exists'] = readable_exists

    selected_type, selected_path = [], []
    for _, row in out.iterrows():
        if bool(row['clean_text_exists']):
            selected_type.append('txt')
            selected_path.append(row['resolved_clean_text_path'])
        elif bool(row['markdown_exists']):
            selected_type.append('md')
            selected_path.append(row['resolved_markdown_path'])
        else:
            selected_type.append('')
            selected_path.append('')

    out['selected_input_type'] = selected_type
    out['selected_input_path'] = selected_path
    out['has_any_canonical_input'] = out['clean_text_exists'] | out['markdown_exists']

    success_gate = out['manifest_clean_success'] if REQUIRE_CLEAN_SUCCESS else pd.Series(True, index=out.index)
    out['fed_to_regex'] = (
        out['manifest_processing_eligible'].fillna(False)
        & success_gate.fillna(False)
        & out['has_any_canonical_input'].fillna(False)
    )

    def gap_reason(row: pd.Series) -> str:
        if not bool(row.get('manifest_processing_eligible', False)):
            return 'not_processing_eligible'
        if REQUIRE_CLEAN_SUCCESS and not bool(row.get('manifest_clean_success', False)):
            return 'clean_success_false'
        if not bool(row.get('clean_text_exists', False)) and not bool(row.get('markdown_exists', False)):
            return 'missing_clean_txt_and_markdown'
        if not bool(row.get('fed_to_regex', False)):
            return 'not_fed_other'
        return ''

    out['regex_gap_reason'] = out.apply(gap_reason, axis=1)
    return out


input_audit = build_input_audit(regex_document_manifest)
regex_inputs = input_audit[input_audit['fed_to_regex']].copy().reset_index(drop=True)
input_gaps = input_audit[~input_audit['fed_to_regex']].copy().reset_index(drop=True)

coverage_rows = []
for source, g in input_audit.groupby('commission_source', dropna=False, sort=False):
    coverage_rows.append({
        'commission_source': source,
        'manifest_documents': int(len(g)),
        'processing_eligible': int(g['manifest_processing_eligible'].fillna(False).sum()),
        'clean_success': int(g['manifest_clean_success'].fillna(False).sum()),
        'clean_txt_exists': int(g['clean_text_exists'].fillna(False).sum()),
        'markdown_exists': int(g['markdown_exists'].fillna(False).sum()),
        'both_txt_and_md_exist': int((g['clean_text_exists'] & g['markdown_exists']).sum()),
        'txt_only': int((g['clean_text_exists'] & ~g['markdown_exists']).sum()),
        'md_only': int((~g['clean_text_exists'] & g['markdown_exists']).sum()),
        'neither_txt_nor_md': int((~g['clean_text_exists'] & ~g['markdown_exists']).sum()),
        'fed_to_regex': int(g['fed_to_regex'].fillna(False).sum()),
        'not_fed_to_regex': int((~g['fed_to_regex'].fillna(False)).sum()),
    })
coverage_by_source = pd.DataFrame(coverage_rows)

coverage_summary = pd.DataFrame([
    {'metric': 'clean_manifest_documents', 'value': int(len(input_audit))},
    {'metric': 'documents_processing_eligible', 'value': int(input_audit['manifest_processing_eligible'].sum())},
    {'metric': 'documents_clean_success', 'value': int(input_audit['manifest_clean_success'].sum())},
    {'metric': 'documents_clean_txt_exists', 'value': int(input_audit['clean_text_exists'].sum())},
    {'metric': 'documents_markdown_exists', 'value': int(input_audit['markdown_exists'].sum())},
    {'metric': 'documents_with_both_txt_and_md', 'value': int((input_audit['clean_text_exists'] & input_audit['markdown_exists']).sum())},
    {'metric': 'documents_with_txt_only', 'value': int((input_audit['clean_text_exists'] & ~input_audit['markdown_exists']).sum())},
    {'metric': 'documents_with_md_only', 'value': int((~input_audit['clean_text_exists'] & input_audit['markdown_exists']).sum())},
    {'metric': 'documents_with_neither_txt_nor_md', 'value': int((~input_audit['clean_text_exists'] & ~input_audit['markdown_exists']).sum())},
    {'metric': 'documents_fed_to_regex', 'value': int(input_audit['fed_to_regex'].sum())},
    {'metric': 'documents_NOT_fed_to_regex', 'value': int((~input_audit['fed_to_regex']).sum())},
])

# Freeze the exact regex universe before scanning.
input_audit.to_csv(INPUT_MANIFEST_CSV, index=False, encoding='utf-8')
with pd.ExcelWriter(INPUT_MANIFEST_XLSX, engine='openpyxl') as writer:
    to_excel_safe(coverage_summary, writer, sheet_name='summary', index=False)
    to_excel_safe(coverage_by_source, writer, sheet_name='coverage_by_source', index=False)
    to_excel_safe(input_audit, writer, sheet_name='all_clean_documents', index=False)
    to_excel_safe(regex_inputs, writer, sheet_name='fed_to_regex', index=False)
    to_excel_safe(input_gaps, writer, sheet_name='NOT_fed_to_regex', index=False)
    to_excel_safe(duplicate_manifest_rows, writer, sheet_name='duplicate_manifest_rows', index=False)
    to_excel_safe(clean_manifest_status, writer, sheet_name='manifest_status', index=False)

print('Coverage by source:')
display(coverage_by_source)
print('Gap reasons:')
display(input_gaps['regex_gap_reason'].value_counts(dropna=False).rename_axis('reason').reset_index(name='n'))
print('Exact documents entering regex:', len(regex_inputs))


Coverage by source:


,commission_source,manifest_documents,processing_eligible,clean_success,clean_txt_exists,markdown_exists,both_txt_and_md_exist,txt_only,md_only,neither_txt_nor_md,fed_to_regex,not_fed_to_regex
0,com_antitrust,1174,1174,1174,1174,1174,1174,0,0,0,1174,0
1,com_db_comp,833,833,833,833,833,833,0,0,0,833,0
2,com_legacy,1505,1505,1504,1505,1505,1505,0,0,0,1504,1
3,com_manual,16,16,16,16,16,16,0,0,0,16,0


Gap reasons:


,reason,n
0,clean_success_false,1


Exact documents entering regex: 3527


## 3. Text-reading helpers


In [4]:
def strip_markdown_frontmatter(text: str) -> str:
    text = text or ''
    if re.match(r'^\ufeff?\s*---\s*\n', text):
        m = re.match(r'^\ufeff?\s*---\s*\n.*?\n---\s*(?:\n|$)', text, flags=re.DOTALL)
        if m:
            return text[m.end():]
    return text


def read_clean_input(path_text: str, input_type: str) -> Tuple[str, str]:
    path = Path(path_text)
    text = path.read_text(encoding='utf-8', errors='replace')
    if input_type == 'md' or path.suffix.lower() == '.md':
        return strip_markdown_frontmatter(text), 'clean_markdown_fallback'
    return text, 'clean_text'


def normalize_text_for_scanning(text: str) -> str:
    text = unicodedata.normalize('NFKC', text or '')
    return text.replace('\r\n', '\n').replace('\r', '\n').replace('\u00a0', ' ')


def compact_whitespace(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text or '')).strip()


def paragraph_spans(text: str) -> List[Tuple[int, int, str]]:
    spans = []
    for match in re.finditer(r'(?s)(?:^|\n\s*\n)(.*?)(?=\n\s*\n|$)', text):
        paragraph = match.group(1).strip()
        if not paragraph:
            continue
        if len(paragraph) > MAX_PARAGRAPH_CHARS:
            paragraph = paragraph[:MAX_PARAGRAPH_CHARS] + ' [...]'
        spans.append((match.start(1), match.end(1), paragraph))
    return spans


def find_paragraph_for_match(spans: List[Tuple[int, int, str]], start: int, end: int) -> str:
    for para_start, para_end, paragraph in spans:
        if para_start <= start and end <= para_end:
            return paragraph
    for para_start, para_end, paragraph in spans:
        if para_start <= start <= para_end:
            return paragraph
    return ''


def safe_context(text: str, start: int, end: int, window: int = CONTEXT_WINDOW_CHARS) -> str:
    return text[max(0, start-window):min(len(text), end+window)]


## 4. Regex definitions

The regex logic is split into exactly three semantic families:

- **Court cases**: explicit `C-` / `T-` cases plus historical unprefixed Court case numbers with Court/Case contextual cues.
- **Commission references**: published Commission decision references such as `88/22/EEC`, with multilingual suffix handling and Commission/decision vicinity filters.
- **Commission cases**: internal competition identifiers beginning `COMP`, `IV`, or `AT`, such as `IV/33.322`.

`F-` staff cases remain excluded, matching the older Court logic.


In [5]:
# ---------------------------------------------------------------------
# Court regexes
# ---------------------------------------------------------------------

HYPHENS = r"\-‐‑‒–—−"
COURT_SUFFIX_RE = r'(?:P\s*[-‐‑‒–—]?\s*DEP|P|PPU|R|DEP|RX|REV|RENV|OP|EXT)'

EXPLICIT_CT_RE = re.compile(
    rf"""
    (?<![A-Za-z0-9])
    (?P<court_prefix>[CTct])
    \s*[{HYPHENS}]?\s*
    (?P<court_number>\d{{1,4}})
    \s*/\s*
    (?P<court_year>\d{{2}})
    (?:\s+(?P<court_suffix>{COURT_SUFFIX_RE}))?
    (?!\d)
    """,
    re.VERBOSE,
)

LEGACY_BARE_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    (?P<court_number>\d{1,4})
    \s*/\s*
    (?P<court_year>5\d|6\d|7\d|8\d|9[0-8])
    (?!\d)
    """,
    re.VERBOSE,
)

LEGACY_CASE_CUE_RE = re.compile(r"\b(?:joined\s+)?cases?\b", re.IGNORECASE)
LEGACY_COURT_CUE_RE = re.compile(r"\bcourts?\b", re.IGNORECASE)


def legacy_context_trigger(context: str) -> str:
    has_case = bool(LEGACY_CASE_CUE_RE.search(context))
    has_court = bool(LEGACY_COURT_CUE_RE.search(context))
    if has_case and has_court:
        return 'case_and_court'
    if has_case:
        return 'case'
    if has_court:
        return 'court'
    return ''


def normalize_court_match(match: re.Match, explicit: bool) -> str:
    gd = match.groupdict()
    prefix = (gd.get('court_prefix') or '').upper()
    number_raw = gd.get('court_number') or ''
    year = gd.get('court_year') or ''
    number = str(int(number_raw)) if number_raw.isdigit() else number_raw
    if explicit:
        return f'{prefix}-{number}/{year}'
    return f'{number}/{year}'


# ---------------------------------------------------------------------
# Commission internal competition case references
# ---------------------------------------------------------------------

COMMISSION_PUNCT_SEP_CHARS = r"[-‐‑‒–—−/\\._]"
COMMISSION_PUNCT_SEP_RE = rf"(?:{COMMISSION_PUNCT_SEP_CHARS})"
COMMISSION_CODE_COMPONENT_RE = r"(?:[A-Z]{1,3}\d{0,8}|\d{1,8}[A-Z]{0,3})"
COMMISSION_BODY_JOIN_RE = rf"(?:\s*{COMMISSION_PUNCT_SEP_RE}\s*|\s+)"
COMMISSION_BODY_AFTER_PUNCT_RE = rf"""
    {COMMISSION_CODE_COMPONENT_RE}
    (?:
        {COMMISSION_BODY_JOIN_RE}
        {COMMISSION_CODE_COMPONENT_RE}
    )*
"""

COMMISSION_CASE_RE = re.compile(
    rf"""(?x)
    (?<![A-Za-z0-9])
    (?P<commission_prefix>COMP|IV|AT)
    (?=(?:[A-Z0-9\s]|{COMMISSION_PUNCT_SEP_CHARS}){{0,40}}\d{{2,}})
    \s*(?P<commission_initial_punct>{COMMISSION_PUNCT_SEP_RE})\s*
    (?P<commission_body_punct>{COMMISSION_BODY_AFTER_PUNCT_RE})
    (?![A-Za-z0-9])
    """
)


def normalize_commission_case(match: re.Match) -> str:
    raw = unicodedata.normalize('NFKC', match.group(0))
    raw = re.sub(r'[‐‑‒–—−]', '-', raw.upper())
    raw = re.sub(r'\s*([/\\._\-])\s*', r'\1', raw)
    raw = re.sub(r'(?<=[A-Z])\s+(?=\d)', '', raw)
    raw = re.sub(r'(?<=\d)\s+(?=[A-Z])', '', raw)
    return compact_whitespace(raw)


# ---------------------------------------------------------------------
# Published Commission decision references
# ---------------------------------------------------------------------

DECISION_REF_SEP_CHARS = r"[/\\_\-‐‑‒–—−]"
DECISION_INNER_NOISE = r"['’_\-‐‑‒–—−]?"
DECISION_YEAR_RE = rf"(?:\d(?:\s*{DECISION_INNER_NOISE}\s*\d){{1}}|\d(?:\s*{DECISION_INNER_NOISE}\s*\d){{3}})"
DECISION_NUMBER_RE = rf"\d(?:\s*{DECISION_INNER_NOISE}\s*\d){{0,3}}"

DECISION_SUFFIX_CANONICAL_MAP = {
    'ECSC': 'ECSC', 'CECA': 'ECSC', 'EGKS': 'ECSC', 'EKSF': 'ECSC', 'EKSG': 'ECSC',
    'EHTY': 'ECSC', 'ΕΚΑΧ': 'ECSC', 'ЕОВС': 'ECSC', 'ESUO': 'ECSC', 'ESTÜ': 'ECSC',
    'EOTK': 'ECSC', 'EAPB': 'ECSC', 'ESZAK': 'ECSC', 'KEFA': 'ECSC', 'EWWIS': 'ECSC',
    'CECO': 'ECSC', 'ESPJ': 'ECSC', 'EZUČ': 'ECSC', 'CEGC': 'ECSC',
    'EEC': 'EEC', 'CEE': 'EEC', 'EWG': 'EEC', 'EEG': 'EEC', 'EØF': 'EEC', 'ETY': 'EEC',
    'ΕΟΚ': 'EEC', 'ЕИО': 'EEC', 'EHS': 'EEC', 'EMÜ': 'EEC', 'EEK': 'EEC', 'EEB': 'EEC',
    'EGK': 'EEC', 'KEE': 'EEC', 'EGS': 'EEC', 'EEZ': 'EEC',
    'EC': 'EC', 'CE': 'EC', 'EG': 'EC', 'EF': 'EC', 'EY': 'EC', 'ΕΚ': 'EC', 'ЕО': 'EC',
    'EÜ': 'EC', 'EK': 'EC', 'EB': 'EC', 'KE': 'EC', 'WE': 'EC', 'EZ': 'EC',
    'EU': 'EU', 'UE': 'EU', 'ΕΕ': 'EU', 'ЕС': 'EU', 'EÚ': 'EU', 'EL': 'EU',
    'AE': 'EU', 'ES': 'ES'
}
KNOWN_DECISION_SUFFIXES = sorted(DECISION_SUFFIX_CANONICAL_MAP, key=len, reverse=True)


def _known_suffix_to_regex(suffix: str) -> str:
    chars = [re.escape(ch) for ch in suffix]
    return r"[.\s]*".join(chars)


DECISION_SUFFIX_RE = '(?:' + '|'.join(_known_suffix_to_regex(s) for s in KNOWN_DECISION_SUFFIXES) + ')'

COMMISSION_TERM_PATTERN = r"""(?:
    commission|kommission|commissione|comisión|comissão|commissie|
    komissio|komise|komisij|komisj|komisjon|komisijos|komisijas|
    bizottság|comisi|комис|επιτροπ|kummissjon|coimisi
)"""
DECISION_TERM_PATTERN = r"""(?:
    decision|décision|entscheid|beschluss|decisione|decisión|decisão|
    beschikking|besluit|afgørelse|beslut|päät|απόφασ|решен|rozhod|
    sklep|decyzj|otsus|lēmum|sprend|határoz|decizi|odluk|deċiż|cinneadh
)"""
COUNCIL_TERM_PATTERN = r"""(?:
    council|conseil|rat|consiglio|consejo|conselho|raad|rådet|neuvosto|
    συμβούλιο|совет|rada|taryba|tanács|consiliu
)"""

COMMISSION_TERM_RE = re.compile(COMMISSION_TERM_PATTERN, re.IGNORECASE | re.VERBOSE | re.UNICODE)
DECISION_TERM_RE = re.compile(DECISION_TERM_PATTERN, re.IGNORECASE | re.VERBOSE | re.UNICODE)
COUNCIL_TERM_RE = re.compile(COUNCIL_TERM_PATTERN, re.IGNORECASE | re.VERBOSE | re.UNICODE)
DECISION_CONTEXT_BOUNDARY_RE = re.compile(r'[.;!?]')

COMMISSION_DECISION_TERM_GAP = 30
DECISION_REFERENCE_TERM_GAP = 50

PUBLISHED_DECISION_SUFFIXED_RE = re.compile(
    rf"""(?<!\d)
    (?P<decision_year>{DECISION_YEAR_RE})
    \s*{DECISION_REF_SEP_CHARS}\s*
    (?P<decision_number>{DECISION_NUMBER_RE})
    \s*{DECISION_REF_SEP_CHARS}\s*
    (?P<decision_suffix>{DECISION_SUFFIX_RE})
    (?![\w])""",
    re.IGNORECASE | re.VERBOSE,
)

PUBLISHED_DECISION_SUFFIXLESS_RE = re.compile(
    rf"""(?<!\d)
    (?P<decision_year>{DECISION_YEAR_RE})
    \s*{DECISION_REF_SEP_CHARS}\s*
    (?P<decision_number>{DECISION_NUMBER_RE})
    (?!\s*{DECISION_REF_SEP_CHARS}\s*(?:{DECISION_SUFFIX_RE}))
    (?!\d)""",
    re.IGNORECASE | re.VERBOSE,
)


In [6]:
def _digits_only(value: str) -> str:
    return re.sub(r'\D', '', unicodedata.normalize('NFKC', str(value or '')))


def normalize_decision_suffix(value: str) -> str:
    value = unicodedata.normalize('NFKC', str(value or ''))
    return ''.join(ch for ch in value if ch.isalpha()).upper()


def canonicalize_decision_suffix(value: str) -> str:
    return DECISION_SUFFIX_CANONICAL_MAP.get(normalize_decision_suffix(value), '')


def is_known_decision_suffix(value: str) -> bool:
    return bool(canonicalize_decision_suffix(value))


def expand_decision_year(year_digits: str) -> Optional[int]:
    digits = _digits_only(year_digits)
    if len(digits) == 4:
        return int(digits)
    if len(digits) == 2:
        yy = int(digits)
        return 1900 + yy if yy >= 52 else 2000 + yy
    return None


def normalize_published_decision(match: re.Match) -> Tuple[str, str]:
    gd = match.groupdict()
    year_digits = _digits_only(gd.get('decision_year') or '')
    number_digits = _digits_only(gd.get('decision_number') or '')
    suffix_original = normalize_decision_suffix(gd.get('decision_suffix') or '')
    suffix_canonical = canonicalize_decision_suffix(suffix_original) if suffix_original else ''
    full_year = expand_decision_year(year_digits)

    if full_year is None or not number_digits or not 1 <= len(number_digits) <= 4 or (suffix_original and not suffix_canonical):
        return compact_whitespace(match.group(0)), ''

    number = int(number_digits)
    display_year = f'{full_year % 100:02d}' if len(year_digits) == 2 else str(full_year)
    normalized_reference = f'{display_year}/{number}'
    if suffix_canonical:
        normalized_reference += f'/{suffix_canonical}'

    derived_celex = f'3{full_year}D{number:04d}'
    return normalized_reference, derived_celex


def is_plausible_published_decision(match: re.Match) -> bool:
    gd = match.groupdict()
    full_year = expand_decision_year(gd.get('decision_year') or '')
    number_digits = _digits_only(gd.get('decision_number') or '')
    suffix = normalize_decision_suffix(gd.get('decision_suffix') or '')
    if full_year is None or not number_digits:
        return False
    return (
        1952 <= full_year <= 2051
        and 1 <= len(number_digits) <= 4
        and 1 <= int(number_digits) <= 9999
        and (not suffix or is_known_decision_suffix(suffix))
    )


def local_bounded_context(text: str, start: int, end: int, radius: int = DECISION_CONTEXT_RADIUS) -> str:
    left = max(0, start - radius)
    right = min(len(text), end + radius)
    left_chunk = text[left:start]
    right_chunk = text[end:right]
    left_matches = list(DECISION_CONTEXT_BOUNDARY_RE.finditer(left_chunk))
    if left_matches:
        left = left + left_matches[-1].end()
    right_match = DECISION_CONTEXT_BOUNDARY_RE.search(right_chunk)
    if right_match:
        right = end + right_match.start()
    return text[left:right]


def has_commission_decision_vicinity(text: str, match: re.Match) -> bool:
    left = max(0, match.start() - COMMISSION_DECISION_CONTEXT_CHARS)
    right = min(len(text), match.end() + COMMISSION_DECISION_CONTEXT_CHARS)
    context = text[left:right]
    return bool(COMMISSION_TERM_RE.search(context) and DECISION_TERM_RE.search(context))


def is_tightly_linked_court_case_cue(text: str, match: re.Match) -> bool:
    left = text[max(0, match.start() - 80):match.start()]
    return bool(
        re.search(r'(?i)\b(?:joined\s+)?cases?(?:\s+nos?\.?)?\s*$', left)
        or re.search(r'(?i)\bjoined\s+cases?\b[^.;!?]{0,60}(?:and\s*)?$', left)
    )


def is_tightly_linked_council_decision(text: str, match: re.Match) -> bool:
    context = local_bounded_context(text, match.start(), match.end())
    located = text.find(context, max(0, match.start() - DECISION_CONTEXT_RADIUS), match.end() + DECISION_CONTEXT_RADIUS)
    if located < 0:
        located = max(0, match.start() - DECISION_CONTEXT_RADIUS)

    candidate_start = match.start() - located
    candidate_end = match.end() - located
    council_spans = [m.span() for m in COUNCIL_TERM_RE.finditer(context)]
    decision_spans = [m.span() for m in DECISION_TERM_RE.finditer(context)]

    for cs, ce in council_spans:
        for ds, de in decision_spans:
            if ce <= ds <= de <= candidate_start:
                if ds - ce <= COMMISSION_DECISION_TERM_GAP and candidate_start - de <= DECISION_REFERENCE_TERM_GAP:
                    return True
            if de <= candidate_start <= candidate_end <= cs:
                if candidate_start - de <= DECISION_REFERENCE_TERM_GAP and cs - candidate_end <= DECISION_REFERENCE_TERM_GAP:
                    return True
            if de <= cs <= ce <= candidate_start:
                if cs - de <= COMMISSION_DECISION_TERM_GAP and candidate_start - ce <= DECISION_REFERENCE_TERM_GAP:
                    return True
    return False


def looks_like_published_decision_suffix_after(text: str, end: int) -> bool:
    # Prevent historical Court regex from taking the year/number core of e.g. 80/234/EEC.
    after = text[end:min(len(text), end + 20)]
    return bool(re.match(rf'\s*{DECISION_REF_SEP_CHARS}\s*(?:{DECISION_SUFFIX_RE})', after, re.IGNORECASE))


def make_hit_id(manifest_row_id: int, source_file: str, start: int, end: int, hit_type: str, matched: str) -> str:
    raw = f'{manifest_row_id}|{source_file}|{start}|{end}|{hit_type}|{matched}'
    return hashlib.sha1(raw.encode('utf-8', errors='replace')).hexdigest()[:16]


def manifest_case_tokens(value: Any) -> set[str]:
    text = clean_scalar(value).upper()
    if not text:
        return set()
    # Manifest case_numbers are commonly semicolon-separated and may contain letter suffixes.
    tokens = set()
    for part in re.split(r'[;|,]+', text):
        token = re.sub(r'[^A-Z0-9]', '', part)
        if token:
            token = token.lstrip('0') or '0'
            tokens.add(token)
    return tokens


def commission_candidate_tokens(normalized: str) -> set[str]:
    text = re.sub(r'^(?:COMP|IV|AT)', '', str(normalized or '').upper())
    components = [re.sub(r'[^A-Z0-9]', '', x) for x in re.split(r'[/\\._\-\s]+', text) if x]
    collapsed = re.sub(r'[^A-Z0-9]', '', text)
    tokens = {x.lstrip('0') or '0' for x in components if x}
    if collapsed:
        tokens.add(collapsed.lstrip('0') or '0')
    # Also add digit/letter tails such as 33133A.
    tail = re.search(r'(\d{2,8}[A-Z]{0,3})$', collapsed)
    if tail:
        tokens.add(tail.group(1).lstrip('0') or '0')
    return tokens


In [7]:
def source_case_tokens(row: pd.Series) -> set[str]:
    """Tokens describing the current source document's own Commission case number(s)."""
    text = clean_scalar(row.get('source_case_numbers', '')).upper()
    if not text:
        return set()
    tokens = set()
    for part in re.split(r'[;|,]+', text):
        token = re.sub(r'[^A-Z0-9]', '', part)
        if token:
            tokens.add(token.lstrip('0') or '0')
    return tokens


def commission_candidate_tokens(normalized: str) -> set[str]:
    text = re.sub(r'^(?:COMP|IV|AT)', '', str(normalized or '').upper())
    components = [re.sub(r'[^A-Z0-9]', '', x) for x in re.split(r'[/\\._\-\s]+', text) if x]
    collapsed = re.sub(r'[^A-Z0-9]', '', text)
    tokens = {x.lstrip('0') or '0' for x in components if x}
    if collapsed:
        tokens.add(collapsed.lstrip('0') or '0')
    tail = re.search(r'(\d{2,8}[A-Z]{0,3})$', collapsed)
    if tail:
        tokens.add(tail.group(1).lstrip('0') or '0')
    return tokens


def source_celex_candidates(row: pd.Series) -> set[str]:
    """Find CELEX decision identifiers in explicit metadata and source/path fields."""
    likely_cols = [
        'source_celex', 'celex', 'celex_id', 'celex_number', 'identifier',
        'source_url', 'source_url_for_processing', 'download_url', 'raw_file_path'
    ]
    haystack = ' | '.join(clean_scalar(row.get(c, '')) for c in likely_cols if c in row.index)
    return {m.upper() for m in re.findall(r'(?i)\b3\d{4}D\d{4}\b', haystack)}


def make_hit_id(source_document_key: str, source_file: str, start: int, end: int, hit_type: str, matched: str) -> str:
    raw = f'{source_document_key}|{source_file}|{start}|{end}|{hit_type}|{matched}'
    return hashlib.sha1(raw.encode('utf-8', errors='replace')).hexdigest()[:16]


## 5. Regex self-tests


In [8]:
self_test_text = """
In Case C-3/06 P, the Court referred to Case 6/64 and Case T-342/99.
The Commission adopted Decision 88/22/EEC in Case IV/33.322.
The Commission later adopted Decision 2004/337/EC in Case COMP/37.792 Microsoft.
It also mentioned case AT.40099.
Council Decision 2001/123/EC should not be treated as a Commission reference.
"""

# 1. Court cases
court_explicit = [normalize_court_match(m, explicit=True) for m in EXPLICIT_CT_RE.finditer(self_test_text)]
court_bare = []
for m in LEGACY_BARE_RE.finditer(self_test_text):
    if looks_like_published_decision_suffix_after(self_test_text, m.end()):
        continue
    if legacy_context_trigger(safe_context(self_test_text, *m.span())):
        court_bare.append(normalize_court_match(m, explicit=False))

assert 'C-3/06' in court_explicit
assert 'T-342/99' in court_explicit
assert '6/64' in court_bare

# 2. Commission references
sample_ref = 'The Commission adopted Decision 88/22/EEC.'
m_ref = PUBLISHED_DECISION_SUFFIXED_RE.search(sample_ref)
assert m_ref is not None
assert has_commission_decision_vicinity(sample_ref, m_ref)
normalized_ref, _ = normalize_published_decision(m_ref)
assert normalized_ref == '88/22/EEC'

sample_council = 'The Council adopted Decision 2001/123/EC.'
m_council = PUBLISHED_DECISION_SUFFIXED_RE.search(sample_council)
assert m_council is not None
assert not has_commission_decision_vicinity(sample_council, m_council)

# 3. Commission cases
for sample, expected in {
    'IV/33.322': 'IV/33.322',
    'COMP/37.792': 'COMP/37.792',
    'AT.40099': 'AT.40099',
}.items():
    m_case = COMMISSION_CASE_RE.search(sample)
    assert m_case is not None, sample
    assert normalize_commission_case(m_case) == expected

# A published decision reference must not be stolen by the historical Court regex.
m = re.search(r'88/22', self_test_text)
assert m is not None and looks_like_published_decision_suffix_after(self_test_text, m.end())

print('Regex self-tests passed for Court cases, Commission references and Commission cases.')


Regex self-tests passed for Court cases, Commission references and Commission cases.


## 6. Scan every canonical Commission document

Raw hits preserve exact character offsets, context, paragraph and extraction provenance.

Each hit belongs to exactly one requested category:

- `court_case`
- `commission_reference`
- `commission_case`


In [9]:
BASE_META_COLUMNS = [c for c in [
    'regex_document_id', 'source_document_key', 'commission_source', 'document_id',
    'source_native_id', 'source_case_numbers', 'source_celex', 'source_date', 'source_title',
    'raw_file_path', 'file_format', 'selected_method', 'quality_score', 'quality_flag',
    'clean_manifest_path'
] if c in regex_inputs.columns]

HIT_COLUMNS = [
    'hit_id', 'reference_category', 'reference_type', 'hit_type',
    'matched_text', 'normalized_candidate', 'derived_celex', 'court_suffix',
    'confidence_precheck', 'needs_validation', 'is_source_self_reference',
    'context_trigger', 'start_char', 'end_char', 'context', 'paragraph_text',
    'source_file', 'source_file_kind', 'extraction_method', 'text_n_chars',
    *BASE_META_COLUMNS
]


def base_hit_record(row: pd.Series, source_file: str, extraction_method: str, text_n_chars: int) -> Dict[str, Any]:
    rec = {col: row.get(col, '') for col in BASE_META_COLUMNS}
    rec.update({
        'source_file': source_file,
        'source_file_kind': row.get('selected_input_type', ''),
        'extraction_method': extraction_method,
        'text_n_chars': text_n_chars,
    })
    return rec


def scan_text(row: pd.Series, text: str, extraction_method: str) -> List[Dict[str, Any]]:
    text = normalize_text_for_scanning(text)
    spans = paragraph_spans(text)
    source_file = clean_scalar(row.get('selected_input_path', ''))
    source_key = clean_scalar(row.get('source_document_key', ''))
    base = base_hit_record(row, source_file, extraction_method, len(text))
    hits: List[Dict[str, Any]] = []

    own_case_tokens = source_case_tokens(row)
    own_celex = source_celex_candidates(row)

    # -------------------------------------------------------------
    # A. Court cases: explicit C/T cases
    # -------------------------------------------------------------
    explicit_spans = []
    for m in EXPLICIT_CT_RE.finditer(text):
        start, end = m.span()
        explicit_spans.append((start, end))
        matched = text[start:end]
        normalized = normalize_court_match(m, explicit=True)
        suffix = compact_whitespace(m.groupdict().get('court_suffix') or '')
        hits.append({
            **base,
            'hit_id': make_hit_id(source_key, source_file, start, end, 'court_explicit_CT', matched),
            'reference_category': 'court_case',
            'reference_type': 'court_case_number',
            'hit_type': 'court_explicit_CT',
            'matched_text': compact_whitespace(matched),
            'normalized_candidate': normalized,
            'derived_celex': '',
            'court_suffix': suffix,
            'confidence_precheck': 'high',
            'needs_validation': True,
            'is_source_self_reference': False,
            'context_trigger': 'explicit_prefix',
            'start_char': start,
            'end_char': end,
            'context': safe_context(text, start, end),
            'paragraph_text': find_paragraph_for_match(spans, start, end),
        })

    # Historical unprefixed Court cases.
    for m in LEGACY_BARE_RE.finditer(text):
        start, end = m.span()
        if any(max(start, s) < min(end, e) for s, e in explicit_spans):
            continue
        if looks_like_published_decision_suffix_after(text, end):
            continue
        context = safe_context(text, start, end)
        trigger = legacy_context_trigger(context)
        if not trigger:
            continue
        matched = text[start:end]
        normalized = normalize_court_match(m, explicit=False)
        hits.append({
            **base,
            'hit_id': make_hit_id(source_key, source_file, start, end, 'court_legacy_unprefixed', matched),
            'reference_category': 'court_case',
            'reference_type': 'court_case_number',
            'hit_type': 'court_legacy_unprefixed',
            'matched_text': compact_whitespace(matched),
            'normalized_candidate': normalized,
            'derived_celex': '',
            'court_suffix': '',
            'confidence_precheck': 'medium',
            'needs_validation': True,
            'is_source_self_reference': False,
            'context_trigger': trigger,
            'start_char': start,
            'end_char': end,
            'context': context,
            'paragraph_text': find_paragraph_for_match(spans, start, end),
        })

    # -------------------------------------------------------------
    # B. Commission cases: internal COMP / IV / AT identifiers
    # -------------------------------------------------------------
    for m in COMMISSION_CASE_RE.finditer(text):
        if not re.search(r'\d', m.group(0)):
            continue
        start, end = m.span()
        matched = text[start:end]
        normalized = normalize_commission_case(m)
        candidate_tokens = commission_candidate_tokens(normalized)
        is_self = bool(own_case_tokens & candidate_tokens)
        hits.append({
            **base,
            'hit_id': make_hit_id(source_key, source_file, start, end, 'commission_case_number_structured', matched),
            'reference_category': 'commission_case',
            'reference_type': 'internal_competition_case_number',
            'hit_type': 'commission_case_number_structured',
            'matched_text': compact_whitespace(matched),
            'normalized_candidate': normalized,
            'derived_celex': '',
            'court_suffix': '',
            'confidence_precheck': 'high',
            'needs_validation': True,
            'is_source_self_reference': is_self,
            'context_trigger': 'structured_COMP_IV_AT',
            'start_char': start,
            'end_char': end,
            'context': safe_context(text, start, end),
            'paragraph_text': find_paragraph_for_match(spans, start, end),
        })

    # -------------------------------------------------------------
    # C. Commission references: published Commission decision numbers
    # -------------------------------------------------------------
    for hit_type, pattern, confidence in [
        ('commission_published_decision_reference_suffixed', PUBLISHED_DECISION_SUFFIXED_RE, 'high'),
        ('commission_published_decision_reference_suffixless_anchored', PUBLISHED_DECISION_SUFFIXLESS_RE, 'medium'),
    ]:
        for m in pattern.finditer(text):
            if not is_plausible_published_decision(m):
                continue
            if not has_commission_decision_vicinity(text, m):
                continue
            if is_tightly_linked_council_decision(text, m):
                continue
            if is_tightly_linked_court_case_cue(text, m):
                continue

            start, end = m.span()
            matched = text[start:end]
            normalized, derived_celex = normalize_published_decision(m)
            is_self = bool(derived_celex and derived_celex.upper() in own_celex)
            hits.append({
                **base,
                'hit_id': make_hit_id(source_key, source_file, start, end, hit_type, matched),
                'reference_category': 'commission_reference',
                'reference_type': 'published_decision_number',
                'hit_type': hit_type,
                'matched_text': compact_whitespace(matched),
                'normalized_candidate': normalized,
                'derived_celex': derived_celex,
                'court_suffix': '',
                'confidence_precheck': confidence,
                'needs_validation': True,
                'is_source_self_reference': is_self,
                'context_trigger': 'commission_and_decision_vicinity',
                'start_char': start,
                'end_char': end,
                'context': local_bounded_context(text, start, end),
                'paragraph_text': find_paragraph_for_match(spans, start, end),
            })

    return hits


In [10]:
all_hits: List[Dict[str, Any]] = []
input_read_rows: List[Dict[str, Any]] = []

for _, row in tqdm(regex_inputs.iterrows(), total=len(regex_inputs), desc='Scanning cleaned Commission files'):
    input_file = clean_scalar(row.get('selected_input_path', ''))
    input_type = clean_scalar(row.get('selected_input_type', ''))

    read_rec = {
        'regex_document_id': row.get('regex_document_id', ''),
        'source_document_key': row.get('source_document_key', ''),
        'commission_source': row.get('commission_source', ''),
        'document_id': row.get('document_id', ''),
        'source_case_numbers': row.get('source_case_numbers', ''),
        'source_title': row.get('source_title', ''),
        'regex_input_file': input_file,
        'regex_input_type': input_type,
        'input_read_success': False,
        'input_read_method': '',
        'text_n_chars': 0,
        'input_read_error': '',
    }

    try:
        text, read_method = read_clean_input(input_file, input_type)
        text = normalize_text_for_scanning(text)
        read_rec['input_read_method'] = read_method
        read_rec['text_n_chars'] = len(text)
        read_rec['input_read_success'] = len(text.strip()) >= MIN_TEXT_CHARS

        if read_rec['input_read_success']:
            all_hits.extend(scan_text(row, text, read_method))
        else:
            read_rec['input_read_error'] = f'clean_input_below_{MIN_TEXT_CHARS}_chars'
    except Exception as exc:
        read_rec['input_read_error'] = repr(exc)

    input_read_rows.append(read_rec)

hits_df = pd.DataFrame(all_hits)
for col in HIT_COLUMNS:
    if col not in hits_df.columns:
        hits_df[col] = pd.NA
hits_df = hits_df[HIT_COLUMNS] if len(hits_df) else pd.DataFrame(columns=HIT_COLUMNS)

input_read_audit = pd.DataFrame(input_read_rows)
runtime_input_gap = (
    input_read_audit[~input_read_audit['input_read_success'].fillna(False)].copy()
    if len(input_read_audit) else pd.DataFrame()
)

hits_df.to_csv(ALL_RAW_HITS_CSV, index=False, encoding='utf-8')

print('Clean inputs attempted:', len(input_read_audit))
print('Clean inputs successfully read:', int(input_read_audit['input_read_success'].sum()) if len(input_read_audit) else 0)
print('Runtime clean-input gap:', len(runtime_input_gap))
print('Total raw regex hits:', len(hits_df))
if len(hits_df):
    display(hits_df.groupby(['reference_category', 'reference_type'], dropna=False).size().reset_index(name='raw_hits'))


Scanning cleaned Commission files:   0%|          | 0/3527 [00:00<?, ?it/s]

Clean inputs attempted: 3527
Clean inputs successfully read: 3527
Runtime clean-input gap: 0
Total raw regex hits: 57530


,reference_category,reference_type,raw_hits
0,commission_case,internal_competition_case_number,14362
1,commission_reference,published_decision_number,4121
2,court_case,court_case_number,39047


## 7. Build the three requested datasets

### Court cases
One row per `source_document_key × normalized Court case`.

Examples: `6/64`, `85/76`, `C-413/14`, `T-342/99`.

### Commission references
One row per `source_document_key × normalized published Commission decision reference`.

Examples: `88/22/EEC`, `2003/706/EC`.

### Commission cases
One row per `source_document_key × normalized Commission competition case identifier`.

Examples: `IV/33.322`, `COMP/37.792`, `AT.39740`.

Multiple mentions inside the same source document are aggregated. Raw occurrence-level hits remain available separately.


In [11]:
def unique_join_series(values: pd.Series, max_values: int = MAX_UNIQUE_VALUES) -> str:
    return unique_join(values.dropna().astype(str).tolist(), max_values=max_values)


def deduplicate_document_identifiers(source_hits: pd.DataFrame) -> pd.DataFrame:
    """One row per source Commission document × normalized identifier."""
    if source_hits.empty:
        return pd.DataFrame(columns=[
            'source_document_key', 'commission_source', 'document_id', 'reference_category',
            'reference_type', 'normalized_candidate', 'n_hits', 'matched_texts', 'hit_types',
            'derived_celex_values', 'court_suffixes', 'contexts', 'paragraphs'
        ])

    rows = []
    group_cols = ['source_document_key', 'reference_type', 'normalized_candidate']
    for (_, _, normalized), g in source_hits.groupby(group_cols, sort=False, dropna=False):
        first = g.sort_values(['source_file', 'start_char', 'end_char']).iloc[0]
        rec = {col: first.get(col, '') for col in BASE_META_COLUMNS}
        rec.update({
            'reference_category': first.get('reference_category', ''),
            'reference_type': first.get('reference_type', ''),
            'normalized_candidate': normalized,
            'n_hits': int(len(g)),
            'matched_texts': unique_join_series(g['matched_text']),
            'hit_types': unique_join_series(g['hit_type']),
            'derived_celex_values': unique_join_series(g['derived_celex']),
            'court_suffixes': unique_join_series(g['court_suffix']),
            'confidence_precheck': unique_join_series(g['confidence_precheck']),
            'is_source_self_reference': bool(g['is_source_self_reference'].fillna(False).any()),
            'first_hit_start': first.get('start_char', ''),
            'first_hit_end': first.get('end_char', ''),
            'source_files': unique_join_series(g['source_file']),
            'contexts': unique_join_series(g['context']),
            'paragraphs': unique_join_series(g['paragraph_text']),
        })
        rows.append(rec)

    return pd.DataFrame(rows).sort_values(
        ['commission_source', 'document_id', 'reference_type', 'normalized_candidate']
    ).reset_index(drop=True)


# Exactly three regex families / outputs.
court_raw_hits = hits_df[hits_df['reference_category'].eq('court_case')].copy()
commission_reference_raw_hits = hits_df[hits_df['reference_category'].eq('commission_reference')].copy()
commission_case_raw_hits = hits_df[hits_df['reference_category'].eq('commission_case')].copy()

court_cases = deduplicate_document_identifiers(court_raw_hits)
commission_references = deduplicate_document_identifiers(commission_reference_raw_hits)
commission_cases = deduplicate_document_identifiers(commission_case_raw_hits)

print('Court case raw occurrences:', len(court_raw_hits))
print('Court case relation rows:', len(court_cases))
print('Commission reference raw occurrences:', len(commission_reference_raw_hits))
print('Commission reference relation rows:', len(commission_references))
print('Commission case raw occurrences:', len(commission_case_raw_hits))
print('Commission case relation rows:', len(commission_cases))

display(court_cases.head(20))
display(commission_references.head(20))
display(commission_cases.head(20))


Court case raw occurrences: 39047
Court case relation rows: 22655
Commission reference raw occurrences: 4121
Commission reference relation rows: 3371
Commission case raw occurrences: 14362
Commission case relation rows: 7110


,regex_document_id,source_document_key,commission_source,document_id,source_native_id,source_case_numbers,source_celex,source_date,source_title,raw_file_path,...,hit_types,derived_celex_values,court_suffixes,confidence_precheck,is_source_self_reference,first_hit_start,first_hit_end,source_files,contexts,paragraphs
0,3,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,3c4a10b1fbb3dca45889,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,,high,False,746,753,/home/edik/projects/eccjeu/data/processed/com_...,ive practices and dominant positions at its me...,× Skip to main content You are here EUROPA EUR...
1,3,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,3c4a10b1fbb3dca45889,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,,high,False,1952,1959,/home/edik/projects/eccjeu/data/processed/com_...,V LT HU MT NL PL PT RO SK SL FI SV Official Jo...,× Skip to main content You are here EUROPA EUR...
2,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_legacy_unprefixed,,,medium,False,131782,131789,/home/edik/projects/eccjeu/data/processed/com_...,# 11.2. Application to this case\n\n(285) The ...,(285) The Commission concludes that it is nece...
3,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_legacy_unprefixed,,,medium,False,75224,75230,/home/edik/projects/eccjeu/data/processed/com_...,"criminatory, but also that it tends to distort...","81 Case 322/81, NV Nederlandsche Banden Indust..."
4,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,,high,False,75323,75331,/home/edik/projects/eccjeu/data/processed/com_...,nder the competitive position of some of the b...,"81 Case 322/81, NV Nederlandsche Banden Indust..."
5,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,P,high,False,75682,75692,/home/edik/projects/eccjeu/data/processed/com_...,", Intel v Commission, EU:T:2014:547, paragraph...","83 Case C-280/08 P, Deutsche Telekom ν Commiss..."
6,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,,high,False,76271,76279,/home/edik/projects/eccjeu/data/processed/com_...,"U:C:1973:22, paragraph 26; Case C-85/76, Hoffm...","86 Case C-311/84, Centre belge d’études de mar..."
7,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,,high,False,76166,76174,/home/edik/projects/eccjeu/data/processed/com_...,"ted therein.\n\n85 Case C-6/72, Europemballage...","85 Case C-6/72, Europemballage Corporation and..."
8,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,cou

,regex_document_id,source_document_key,commission_source,document_id,source_native_id,source_case_numbers,source_celex,source_date,source_title,raw_file_path,...,hit_types,derived_celex_values,court_suffixes,confidence_precheck,is_source_self_reference,first_hit_start,first_hit_end,source_files,contexts,paragraphs
0,5,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,84232b1b83f1fe112993,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffix...,31987D0005,,medium,False,849,854,/home/edik/projects/eccjeu/data/processed/com_...,40703 – Amazon Buy Box) (notified under docume...,× Skip to main content You are here EUROPA EUR...
1,5,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,84232b1b83f1fe112993,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffix...,31987D0007,,medium,False,2257,2261,/home/edik/projects/eccjeu/data/processed/com_...,2023 EN Official Journal of the European Union...,× Skip to main content You are here EUROPA EUR...
2,6,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,a598f86fa3acdd6f04ed,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffixed,32011D0695,,high,False,553,564,/home/edik/projects/eccjeu/data/processed/com_...,Use the Advanced search Document 52023AT40462 ...,× Skip to main content You are here EUROPA EUR...
3,13,com_antitrust||com_antitrust__COM1057__AT.4052...,com_antitrust,com_antitrust__COM1057__AT.40522__12_07_2022__...,c626727c835dcaf7c25b,AT.40522,,12/07/2022,Settlement Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffix...,31957D0004,,medium,False,744,749,/home/edik/projects/eccjeu/data/processed/com_...,40522-Metal packaging) (notified under documen...,× Skip to main content You are here EUROPA EUR...
4,13,com_antitrust||com_antitrust__COM1057__AT.4052...,com_antitrust,com_antitrust__COM1057__AT.40522__12_07_2022__...,c626727c835dcaf7c25b,AT.40522,,12/07/2022,Settlement Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffix...,31957D0005,,medium,False,1959,1963,/home/edik/projects/eccjeu/data/processed/com_...,2023 EN Official Journal of the European Union...,× Skip to main content You are here EUROPA EUR...
5,14,com_antitrust||com_antitrust__COM1057__AT.4052...,com_antitrust,com_antitrust__COM1057__AT.40522__12_07_2022__...,e587c080c2fba5b9ff57,AT.40522,,12/07/2022,Settlement Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffixed,32011D0695,,high,False,561,572,/home/edik/projects/eccjeu/data/processed/com_...,Use the Advanced search Document 52023AT40522(...,× Skip to main content You are here EUROPA EUR...
6,18,com_antitrust||com_antitrust__COM1066__AT.4054...,com_antitrust,com_antitrust__COM1066__AT.40547__29_11_2022__...,c3744b18ddcde065c0eb,AT.40547,,29/11/2022,Settlement Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffixed,32011D0695,,high,False,561,572,/home/edik/projects/eccjeu/data/processed/com_...,Use the Advanced search Document 52023AT40547(...,× Skip to main content You are here EUROPA EUR...
7,21,com_antitrust||com_antitrust__COM1084__AT.4058...,com_antitrust,com_antitrust__COM1084__AT.40588__31_10_2024__...,8629492be8bc0efae7c0,AT.40588,,31/10/2024,Prohibition Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffix...,32016D0017,,medium,False,209951,209956,/home/edik/projects/eccjeu/data/processed/com_...,"9517, paragraphs 16-17 and 20","267 Comm

,regex_document_id,source_document_key,commission_source,document_id,source_native_id,source_case_numbers,source_celex,source_date,source_title,raw_file_path,...,hit_types,derived_celex_values,court_suffixes,confidence_precheck,is_source_self_reference,first_hit_start,first_hit_end,source_files,contexts,paragraphs
0,2,com_antitrust||com_antitrust__COM1000__AT.4046...,com_antitrust,com_antitrust__COM1000__AT.40461__07_12_2018__...,c354a61a3bc8ec627de8,AT.40461,,07/12/2018,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,7,22,/home/edik/projects/eccjeu/data/processed/com_...,# CASE AT.40461 -DE/DK Interconnector\n\n(Only...,# CASE AT.40461 -DE/DK Interconnector | # rela...
1,3,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,3c4a10b1fbb3dca45889,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,634,642,/home/edik/projects/eccjeu/data/processed/com_...,ions? Use the Advanced search Document 52023AT...,× Skip to main content You are here EUROPA EUR...
2,3,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,3c4a10b1fbb3dca45889,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,668,676,/home/edik/projects/eccjeu/data/processed/com_...,ment 52023AT40462(02) Help Print Opinion of th...,× Skip to main content You are here EUROPA EUR...
3,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,7,15,/home/edik/projects/eccjeu/data/processed/com_...,# CASE AT.40462 - Amazon Marketplace and AT.40...,# CASE AT.40462 - Amazon Marketplace and AT.40...
4,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,41,49,/home/edik/projects/eccjeu/data/processed/com_...,# CASE AT.40462 - Amazon Marketplace and AT.40...,# CASE AT.40462 - Amazon Marketplace and AT.40...
5,5,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,84232b1b83f1fe112993,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,673,681,/home/edik/projects/eccjeu/data/processed/com_...,52023AT40462(01) Help Print Summary of Commiss...,× Skip to main content You are here EUROPA EUR...
6,5,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,84232b1b83f1fe112993,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,707,715,/home/edik/projects/eccjeu/data/processed/com_...,y of Commission Decision of 20 December 2022 r...,× Skip to main content You are here EUROPA EUR...
7,6,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,a598f86fa3acdd6f04ed,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,788,796,/home/edik/projects/eccjeu/data/processed/com_...,the President of the European Commission of 13...,× Skip to main content You are here EUROPA EUR...
8,6,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,a598f86fa3acdd6f04ed,AT.40462,,20/12/2022,Commitment Decision,

## 8. Write outputs and final audit


In [12]:
run_info = pd.DataFrame([{
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'notebook_name': NOTEBOOK_NAME,
    'project_root': str(PROJECT_ROOT),
    'require_all_manifests': REQUIRE_ALL_MANIFESTS,
    'require_clean_success': REQUIRE_CLEAN_SUCCESS,
    'input_preference': ' > '.join(INPUT_PREFERENCE),
    'n_clean_manifest_documents': int(len(input_audit)),
    'n_documents_fed_to_regex': int(len(regex_inputs)),
    'n_documents_not_fed_to_regex': int(len(input_gaps)),
    'n_runtime_input_gap': int(len(runtime_input_gap)),
    'n_raw_court_case_hits': int(len(court_raw_hits)),
    'n_court_case_rows': int(len(court_cases)),
    'n_raw_commission_reference_hits': int(len(commission_reference_raw_hits)),
    'n_commission_reference_rows': int(len(commission_references)),
    'n_raw_commission_case_hits': int(len(commission_case_raw_hits)),
    'n_commission_case_rows': int(len(commission_cases)),
}])

final_summary = pd.concat([
    coverage_summary,
    pd.DataFrame([
        {'metric': 'runtime_clean_inputs_failed_to_read', 'value': int(len(runtime_input_gap))},
        {'metric': 'raw_court_case_occurrences', 'value': int(len(court_raw_hits))},
        {'metric': 'court_case_relation_rows', 'value': int(len(court_cases))},
        {'metric': 'raw_commission_reference_occurrences', 'value': int(len(commission_reference_raw_hits))},
        {'metric': 'commission_reference_relation_rows', 'value': int(len(commission_references))},
        {'metric': 'raw_commission_case_occurrences', 'value': int(len(commission_case_raw_hits))},
        {'metric': 'commission_case_relation_rows', 'value': int(len(commission_cases))},
    ])
], ignore_index=True)

with pd.ExcelWriter(AUDIT_XLSX, engine='openpyxl') as writer:
    to_excel_safe(final_summary, writer, sheet_name='summary', index=False)
    to_excel_safe(run_info, writer, sheet_name='run_info', index=False)
    to_excel_safe(clean_manifest_status, writer, sheet_name='manifest_status', index=False)
    to_excel_safe(coverage_by_source, writer, sheet_name='coverage_by_source', index=False)
    to_excel_safe(input_audit, writer, sheet_name='all_clean_documents', index=False)
    to_excel_safe(regex_inputs, writer, sheet_name='fed_to_regex', index=False)
    to_excel_safe(input_gaps, writer, sheet_name='NOT_fed_to_regex', index=False)
    to_excel_safe(input_read_audit, writer, sheet_name='input_read_audit', index=False)
    to_excel_safe(runtime_input_gap, writer, sheet_name='runtime_input_gap', index=False)
    to_excel_safe(duplicate_manifest_rows, writer, sheet_name='duplicate_manifest_rows', index=False)

with pd.ExcelWriter(COURT_CASES_XLSX, engine='openpyxl') as writer:
    to_excel_safe(court_cases, writer, sheet_name='court_cases', index=False)
    to_excel_safe(court_raw_hits, writer, sheet_name='raw_hits', index=False)
    to_excel_safe(run_info, writer, sheet_name='run_info', index=False)

with pd.ExcelWriter(COMMISSION_REFERENCES_XLSX, engine='openpyxl') as writer:
    to_excel_safe(commission_references, writer, sheet_name='commission_references', index=False)
    to_excel_safe(commission_reference_raw_hits, writer, sheet_name='raw_hits', index=False)
    to_excel_safe(run_info, writer, sheet_name='run_info', index=False)

with pd.ExcelWriter(COMMISSION_CASES_XLSX, engine='openpyxl') as writer:
    to_excel_safe(commission_cases, writer, sheet_name='commission_cases', index=False)
    to_excel_safe(commission_case_raw_hits, writer, sheet_name='raw_hits', index=False)
    to_excel_safe(run_info, writer, sheet_name='run_info', index=False)

court_cases.to_csv(COURT_CASES_CSV, index=False, encoding='utf-8')
commission_references.to_csv(COMMISSION_REFERENCES_CSV, index=False, encoding='utf-8')
commission_cases.to_csv(COMMISSION_CASES_CSV, index=False, encoding='utf-8')

print('Wrote input manifest:', INPUT_MANIFEST_XLSX)
print('Wrote audit:', AUDIT_XLSX)
print('Wrote Court cases:', COURT_CASES_XLSX)
print('Wrote Commission references:', COMMISSION_REFERENCES_XLSX)
print('Wrote Commission cases:', COMMISSION_CASES_XLSX)


Wrote input manifest: /home/edik/projects/eccjeu/output/commission_regex/excel/commission_regex_input_manifest.xlsx
Wrote audit: /home/edik/projects/eccjeu/output/commission_regex/excel/commission_regex_audit.xlsx
Wrote Court cases: /home/edik/projects/eccjeu/output/commission_regex/excel/commission_regex_court_cases.xlsx
Wrote Commission references: /home/edik/projects/eccjeu/output/commission_regex/excel/commission_regex_commission_references.xlsx
Wrote Commission cases: /home/edik/projects/eccjeu/output/commission_regex/excel/commission_regex_commission_cases.xlsx


## 9. Final inspection

The two most important integrity checks are:

1. `NOT_fed_to_regex` — clean-manifest documents excluded before scanning, with an explicit reason.
2. `runtime_input_gap` — inputs that existed during the coverage check but could not be read successfully at scan time.

The main regex results are split into exactly three outputs:

- Court cases
- Commission references
- Commission cases


In [13]:
print('=== FINAL RUN SUMMARY ===')
display(final_summary)

print('\n=== COVERAGE BY SOURCE ===')
display(coverage_by_source)

print('\n=== DOCUMENTS NOT FED TO REGEX ===')
display(input_gaps[[c for c in [
    'commission_source', 'document_id', 'source_native_id', 'source_case_numbers',
    'source_title', 'manifest_processing_eligible', 'manifest_clean_success',
    'clean_text_exists', 'markdown_exists', 'regex_gap_reason', 'raw_file_path',
    'clean_text_path', 'markdown_path'
] if c in input_gaps.columns]].head(500))

if len(runtime_input_gap):
    print('\n=== RUNTIME INPUT GAP ===')
    display(runtime_input_gap.head(500))

print('\n=== COURT CASES ===')
display(court_cases.head(100))

print('\n=== COMMISSION REFERENCES ===')
display(commission_references.head(100))

print('\n=== COMMISSION CASES ===')
display(commission_cases.head(100))


=== FINAL RUN SUMMARY ===


,metric,value
0,clean_manifest_documents,3528
1,documents_processing_eligible,3528
2,documents_clean_success,3527
3,documents_clean_txt_exists,3528
4,documents_markdown_exists,3528
5,documents_with_both_txt_and_md,3528
6,documents_with_txt_only,0
7,documents_with_md_only,0
8,documents_with_neither_txt_nor_md,0
9,documents_fed_to_regex,3527



=== COVERAGE BY SOURCE ===


,commission_source,manifest_documents,processing_eligible,clean_success,clean_txt_exists,markdown_exists,both_txt_and_md_exist,txt_only,md_only,neither_txt_nor_md,fed_to_regex,not_fed_to_regex
0,com_antitrust,1174,1174,1174,1174,1174,1174,0,0,0,1174,0
1,com_db_comp,833,833,833,833,833,833,0,0,0,833,0
2,com_legacy,1505,1505,1504,1505,1505,1505,0,0,0,1504,1
3,com_manual,16,16,16,16,16,16,0,0,0,16,0



=== DOCUMENTS NOT FED TO REGEX ===


...

,regex_document_id,source_document_key,commission_source,document_id,source_native_id,source_case_numbers,source_celex,source_date,source_title,raw_file_path,...,hit_types,derived_celex_values,court_suffixes,confidence_precheck,is_source_self_reference,first_hit_start,first_hit_end,source_files,contexts,paragraphs
0,3,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,3c4a10b1fbb3dca45889,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,,high,False,746,753,/home/edik/projects/eccjeu/data/processed/com_...,ive practices and dominant positions at its me...,× Skip to main content You are here EUROPA EUR...
1,3,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,3c4a10b1fbb3dca45889,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,,high,False,1952,1959,/home/edik/projects/eccjeu/data/processed/com_...,V LT HU MT NL PL PT RO SK SL FI SV Official Jo...,× Skip to main content You are here EUROPA EUR...
2,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_legacy_unprefixed,,,medium,False,131782,131789,/home/edik/projects/eccjeu/data/processed/com_...,# 11.2. Application to this case\n\n(285) The ...,(285) The Commission concludes that it is nece...
3,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_legacy_unprefixed,,,medium,False,75224,75230,/home/edik/projects/eccjeu/data/processed/com_...,"criminatory, but also that it tends to distort...","81 Case 322/81, NV Nederlandsche Banden Indust..."
4,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,,high,False,75323,75331,/home/edik/projects/eccjeu/data/processed/com_...,nder the competitive position of some of the b...,"81 Case 322/81, NV Nederlandsche Banden Indust..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,21,com_antitrust||com_antitrust__COM1084__AT.4058...,com_antitrust,com_antitrust__COM1084__AT.40588__31_10_2024__...,8629492be8bc0efae7c0,AT.40588,,31/10/2024,Prohibition Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,,high,False,821387,821395,/home/edik/projects/eccjeu/data/processed/com_...,Judgment of the Court of Justice of 31 October...,1207 Judgment of the Court of Justice of 31 Oc...
96,21,com_antitrust||com_antitrust__COM1084__AT.4058...,com_antitrust,com_antitrust__COM1084__AT.40588__31_10_2024__...,8629492be8bc0efae7c0,AT.40588,,31/10/2024,Prohibition Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,P,high,False,169457,169467,/home/edik/projects/eccjeu/data/processed/com_...,"1645, 22.2.2024, ELI: http://data.europa.eu/el...",222 Commission Notice on the definition of rel...
97,21,com_antitrust||com_antitrust__COM1084__AT.4058...,com_antitrust,com_antitrust__COM1084__AT.40588__31_10_2024__...,8629492be8bc0efae7c0,AT.40588,,31/10/2024,Prohibition Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,court_explicit_CT,,,high,False,842845,842852,/home/edik/projects/eccjeu/data/processed/com_...,"tement of Objections, Section 6.1.1. 1242 Case...",1237 See Section 8.2.1. For rebuttal of Teva’s...
98,21,com_antitrust||com_antitrust__COM1084__AT.4058...,com_antitrust,com_antitrust__COM1084__AT.40588__31_10_2024__...,8629492be8bc0efae7c0,A


=== COMMISSION REFERENCES ===


,regex_document_id,source_document_key,commission_source,document_id,source_native_id,source_case_numbers,source_celex,source_date,source_title,raw_file_path,...,hit_types,derived_celex_values,court_suffixes,confidence_precheck,is_source_self_reference,first_hit_start,first_hit_end,source_files,contexts,paragraphs
0,5,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,84232b1b83f1fe112993,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffix...,31987D0005,,medium,False,849,854,/home/edik/projects/eccjeu/data/processed/com_...,40703 – Amazon Buy Box) (notified under docume...,× Skip to main content You are here EUROPA EUR...
1,5,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,84232b1b83f1fe112993,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffix...,31987D0007,,medium,False,2257,2261,/home/edik/projects/eccjeu/data/processed/com_...,2023 EN Official Journal of the European Union...,× Skip to main content You are here EUROPA EUR...
2,6,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,a598f86fa3acdd6f04ed,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffixed,32011D0695,,high,False,553,564,/home/edik/projects/eccjeu/data/processed/com_...,Use the Advanced search Document 52023AT40462 ...,× Skip to main content You are here EUROPA EUR...
3,13,com_antitrust||com_antitrust__COM1057__AT.4052...,com_antitrust,com_antitrust__COM1057__AT.40522__12_07_2022__...,c626727c835dcaf7c25b,AT.40522,,12/07/2022,Settlement Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffix...,31957D0004,,medium,False,744,749,/home/edik/projects/eccjeu/data/processed/com_...,40522-Metal packaging) (notified under documen...,× Skip to main content You are here EUROPA EUR...
4,13,com_antitrust||com_antitrust__COM1057__AT.4052...,com_antitrust,com_antitrust__COM1057__AT.40522__12_07_2022__...,c626727c835dcaf7c25b,AT.40522,,12/07/2022,Settlement Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffix...,31957D0005,,medium,False,1959,1963,/home/edik/projects/eccjeu/data/processed/com_...,2023 EN Official Journal of the European Union...,× Skip to main content You are here EUROPA EUR...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,74,com_antitrust||com_antitrust__COM190__AT.37133...,com_antitrust,com_antitrust__COM190__AT.37133__23_10_2001__S...,3f9c049f126f50b6ee78,AT.37133,,23/10/2001,State Measure Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffixed,32002D0344,,high,False,478,489,/home/edik/projects/eccjeu/data/processed/com_...,Use the Advanced search Document 32002D0344 He...,× Skip to main content You are here EUROPA EUR...
96,75,com_antitrust||com_antitrust__COM191__AT.37152...,com_antitrust,com_antitrust__COM191__AT.37152__27_11_2002__P...,fbb70b382f51709895ee,AT.37152,,27/11/2002,Prohibition Decision (Art. 7) (Art. 101 & 102 ...,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_decision_reference_suffixed,32002D0405,,high,False,431272,431283,/home/edik/projects/eccjeu/data/processed/com_...,469 Siehe auch Entscheidung 2002/405/EG der Ko...,469 Siehe auch Entscheidung 2002/405/EG der Ko...
97,75,com_antitrust||com_antitrust__COM191__AT.37152...,com_antitrust,com_antitrust__COM191__AT.37152__27_11_2002__P...,fbb70b382f51709895ee,AT.37152,,27/11/2002,Prohibition Decision (Art. 7) (Art. 101 & 102 ...,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_published_


=== COMMISSION CASES ===


,regex_document_id,source_document_key,commission_source,document_id,source_native_id,source_case_numbers,source_celex,source_date,source_title,raw_file_path,...,hit_types,derived_celex_values,court_suffixes,confidence_precheck,is_source_self_reference,first_hit_start,first_hit_end,source_files,contexts,paragraphs
0,2,com_antitrust||com_antitrust__COM1000__AT.4046...,com_antitrust,com_antitrust__COM1000__AT.40461__07_12_2018__...,c354a61a3bc8ec627de8,AT.40461,,07/12/2018,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,7,22,/home/edik/projects/eccjeu/data/processed/com_...,# CASE AT.40461 -DE/DK Interconnector\n\n(Only...,# CASE AT.40461 -DE/DK Interconnector | # rela...
1,3,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,3c4a10b1fbb3dca45889,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,634,642,/home/edik/projects/eccjeu/data/processed/com_...,ions? Use the Advanced search Document 52023AT...,× Skip to main content You are here EUROPA EUR...
2,3,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,3c4a10b1fbb3dca45889,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,668,676,/home/edik/projects/eccjeu/data/processed/com_...,ment 52023AT40462(02) Help Print Opinion of th...,× Skip to main content You are here EUROPA EUR...
3,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,7,15,/home/edik/projects/eccjeu/data/processed/com_...,# CASE AT.40462 - Amazon Marketplace and AT.40...,# CASE AT.40462 - Amazon Marketplace and AT.40...
4,4,com_antitrust||com_antitrust__COM1004__AT.4046...,com_antitrust,com_antitrust__COM1004__AT.40462__20_12_2022__...,835541a3608faa0f7ef9,AT.40462,,20/12/2022,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,41,49,/home/edik/projects/eccjeu/data/processed/com_...,# CASE AT.40462 - Amazon Marketplace and AT.40...,# CASE AT.40462 - Amazon Marketplace and AT.40...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,45,com_antitrust||com_antitrust__COM1178__AT.4087...,com_antitrust,com_antitrust__COM1178__AT.40873__12_09_2025__...,c172c282f2a668930ca8,AT.40873,,12/09/2025,Commitment Decision,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,671,679,/home/edik/projects/eccjeu/data/processed/com_...,## COMMISSION DECISION\n\n## of 12.9.2025\n\nr...,Cases AT.40721 – Microsoft Teams and AT.40873 ...
96,46,com_antitrust||com_antitrust__COM1180__AT.4088...,com_antitrust,com_antitrust__COM1180__AT.40882__24_06_2024__...,0c329405f6f112933aa7,AT.40882,,24/06/2024,Decision imposing fines,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,3948,3956,/home/edik/projects/eccjeu/data/processed/com_...,The consolidated turnover of IFF in 2023 amoun...,"(5) Between 7 and 10 March 2023, the Commissio..."
97,46,com_antitrust||com_antitrust__COM1180__AT.4088...,com_antitrust,com_antitrust__COM1180__AT.40882__24_06_2024__...,0c329405f6f112933aa7,AT.40882,,24/06/2024,Decision imposing fines,/home/edik/projects/eccjeu/data/raw/com_antitr...,...,commission_case_number_structured,,,high,False,32671,32679,/home/edik/projects/eccjeu/data/processed/com_...,"is Decision, to the following bank account hel...",Ref.: EC/BUFI/AT.40882
98,46,com_antitrust||com_antitrust__COM1180__AT.4088...,com_antitrust,com_antitrust__COM1180__A